## Modeling
**Purpose:** Fit reproducible baseline and non-linear models for `U` and `V`, then export standardized artifacts for `04_results_evaluation.ipynb`.  
**Inputs:** `../data/processed/clean_data_train_*.csv` and `../data/processed/clean_data_test_*.csv` (generated in `02_preprocessing.ipynb`).  
**Outputs:** `../data/processed/predictions_u_linear.csv`, `../data/processed/predictions_v_linear.csv`, `../data/processed/predictions_v_rf.csv`, `../data/processed/linear_u_coefficients.csv`, `../data/processed/linear_v_coefficients.csv`, `../data/processed/model_metrics.csv`, and `../data/processed/rf_v_feature_importance.csv`.


In [147]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import aux_functions
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from statstests.process import stepwise
from statstests.tests import shapiro_francia
from aux_functions import evaluate_regression, breusch_pagan

RANDOM_STATE = 100

base = Path('../data/processed')
df_u_train = pd.read_csv(base / 'clean_data_train_u.csv')
df_u_test = pd.read_csv(base / 'clean_data_test_u.csv')
df_v_train = pd.read_csv(base / 'clean_data_train_v.csv')
df_v_test = pd.read_csv(base / 'clean_data_test_v.csv')

print('Loaded cleaned datasets.')

Loaded cleaned datasets.


In [148]:
# One-hot encode using TRAIN dummies as the reference feature set

df_v_train.drop(columns=['Unnamed: 0'], inplace=True)
df_v_test.drop(columns=['Unnamed: 0'], inplace=True)
df_u_train.drop(columns=['Unnamed: 0'], inplace=True)
df_u_test.drop(columns=['Unnamed: 0'], inplace=True)

# --- U ---
u_train_for_dummies = df_u_train.drop(columns=['qmof_id'])
u_test_for_dummies = df_u_test.drop(columns=['qmof_id'])

u_train_dummies = pd.get_dummies(u_train_for_dummies, columns=['atom_i'], dtype=int, drop_first=True)
u_test_dummies = pd.get_dummies(u_test_for_dummies, columns=['atom_i'], dtype=int, drop_first=True)

target_u = 'U'
predictor_cols_u = [c for c in u_train_dummies.columns if c != target_u]

X_train_u = u_train_dummies[predictor_cols_u].copy()
y_train_u = u_train_dummies[target_u].copy()

X_test_u = u_test_dummies.reindex(columns=predictor_cols_u, fill_value=0).copy()
y_test_u = u_test_dummies[target_u].copy()

# --- V ---
v_train_for_dummies = df_v_train.drop(columns=['qmof_id'])
v_test_for_dummies = df_v_test.drop(columns=['qmof_id'])

v_train_dummies = pd.get_dummies(v_train_for_dummies, columns=['atom_i', 'atom_j'], dtype=int, drop_first=True)
v_test_dummies = pd.get_dummies(v_test_for_dummies, columns=['atom_i', 'atom_j'], dtype=int, drop_first=True)

target_v = 'V'
predictor_cols_v = [c for c in v_train_dummies.columns if c != target_v]

X_train_v = v_train_dummies[predictor_cols_v].copy()
y_train_v = v_train_dummies[target_v].copy()

X_test_v = v_test_dummies.reindex(columns=predictor_cols_v, fill_value=0).copy()
y_test_v = v_test_dummies[target_v].copy()

print('U feature columns:', len(predictor_cols_u))
print('V feature columns:', len(predictor_cols_v))

U feature columns: 30
V feature columns: 58


In [149]:
print("For predicting U:")

print(f"Shape of X train data: {X_train_u.shape}")
print(f"Shape of y train data: {y_train_u.shape}")


print(f"Shape of X test data: {X_test_u.shape}")
print(f"Shape of y test data: {y_test_u.shape}")

print("\n")
print("For predicting V:")

print(f"Shape of X train data: {X_train_v.shape}")
print(f"Shape of y train data: {y_train_v.shape}")


print(f"Shape of X test data: {X_test_v.shape}")
print(f"Shape of y test data: {y_test_v.shape}")


For predicting U:
Shape of X train data: (7083, 30)
Shape of y train data: (7083,)
Shape of X test data: (1658, 30)
Shape of y test data: (1658,)


For predicting V:
Shape of X train data: (178924, 58)
Shape of y train data: (178924,)
Shape of X test data: (40927, 58)
Shape of y test data: (40927,)


In [150]:
print('Train dataframe for U dummized:')

print(X_train_u.head(10))


Train dataframe for U dummized:
       pld   density       volume  ddec_charge_i  atom_i_Br  atom_i_C  \
0  3.81269  1.592058  1170.896444       0.234390          0         0   
1  3.81269  1.592058  1170.896444       0.234423          0         0   
2  3.81269  1.592058  1170.896444       0.234366          0         0   
3  3.81269  1.592058  1170.896444      -0.297503          0         0   
4  3.81269  1.592058  1170.896444      -0.297568          0         0   
5  3.81269  1.592058  1170.896444      -0.297412          0         0   
6  3.81269  1.592058  1170.896444      -0.297531          0         0   
7  3.81269  1.592058  1170.896444      -0.389910          0         0   
8  3.81269  1.592058  1170.896444      -0.389836          1         0   
9  3.81269  1.592058  1170.896444      -0.389914          1         0   

   atom_i_Cl  atom_i_Co  atom_i_Cu  atom_i_H  ...  atom_i_Re  atom_i_Rh  \
0          0          0          1         0  ...          0          0   
1          0  

In [151]:
print('Train dataframe for V dummized:')

print(X_train_v.head(10))


Train dataframe for V dummized:
       pld   density       volume  ddec_charge_i  ddec_charge_j   distance  \
0  3.81269  1.592058  1170.896444        0.23439      -0.297568   4.262269   
1  3.81269  1.592058  1170.896444        0.23439      -0.297531   4.519819   
2  3.81269  1.592058  1170.896444        0.23439      -0.389887   4.676169   
3  3.81269  1.592058  1170.896444        0.23439      -0.389836   4.704777   
4  3.81269  1.592058  1170.896444        0.23439       0.234366   5.058731   
5  3.81269  1.592058  1170.896444        0.23439      -0.297568   8.449849   
6  3.81269  1.592058  1170.896444        0.23439      -0.389887   8.588470   
7  3.81269  1.592058  1170.896444        0.23439      -0.389836   8.696982   
8  3.81269  1.592058  1170.896444        0.23439      -0.297531   9.067107   
9  3.81269  1.592058  1170.896444        0.23439       0.234390  10.498118   

   atom_i_Br  atom_i_C  atom_i_Cl  atom_i_Co  ...  atom_j_Re  atom_j_Rh  \
0          0         0          0 

### 1) U linear baseline

Train and diagnose the OLS + stepwise baseline for on-site interaction `U`.

Use this block to:
- verify coefficient significance and residual assumptions,
- inspect train/test fit quality,
- generate the exported `U` linear predictions used in evaluation.


In [152]:
# -----------------------------
# Step 0: Reset indices to ensure alignment
# -----------------------------
X_train_u = X_train_u.reset_index(drop=True)
X_test_u = X_test_u.reset_index(drop=True)
y_train_u = y_train_u.reset_index(drop=True)
y_test_u = y_test_u.reset_index(drop=True)

# Add intercept for statsmodels
X_train_u_const = sm.add_constant(X_train_u)
X_test_u_const = sm.add_constant(X_test_u)

# -----------------------------
# Step 2: Fit initial OLS model
# -----------------------------
modelo_u = sm.OLS(y_train_u, X_train_u_const).fit()
print("Initial OLS Model:")
print(modelo_u.summary())

# -----------------------------
# Step 3: Stepwise selection (p-value threshold 0.05)
# -----------------------------
modelo_step_u = stepwise(modelo_u, pvalue_limit=0.05)
print("Stepwise-selected Model:")
print(modelo_step_u.summary())

# -----------------------------
# Step 4: Residual normality test (Shapiro-Francia)
# -----------------------------
teste_sf = shapiro_francia(modelo_step_u.resid)
teste_sf_items = list(teste_sf.items())
method, statistics_W, statistics_z, p = teste_sf_items
print('Shapiro-Francia Test: Statistics W=%.5f, p-value=%.6f' % (statistics_W[1], p[1]))

alpha = 0.05
if p[1] > alpha:
    print('Não se rejeita H0 - Distribuição aderente à normalidade')
else:
    print('Rejeita-se H0 - Distribuição não aderente à normalidade')




Initial OLS Model:
                            OLS Regression Results                            
Dep. Variable:                      U   R-squared:                       0.974
Model:                            OLS   Adj. R-squared:                  0.974
Method:                 Least Squares   F-statistic:                     8973.
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:49:21   Log-Likelihood:                -1446.6
No. Observations:                7083   AIC:                             2955.
Df Residuals:                    7052   BIC:                             3168.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            10.4091   

In [153]:
def _exog_name_to_df_col(name: str) -> str:
    """Align statsmodels/patsy exog names with DataFrame columns from sm.add_constant + dummies."""
    clean = str(name).replace("Q('", "").replace("')", "")
    # add_constant() creates 'const'; formula/stepwise output often labels the intercept 'Intercept'
    if clean == "Intercept":
        return "const"
    return clean


selected_cols_u = [_exog_name_to_df_col(n) for n in modelo_step_u.model.exog_names]

X_train_u_sel = X_train_u_const[selected_cols_u].copy()
X_test_u_sel = X_test_u_const[selected_cols_u].copy()


metrics_u = evaluate_regression(modelo_step_u, X_train_u_sel, y_train_u, X_test_u_sel, y_test_u)

Training RMSE: 0.2968
Training R²: 0.9745
Training MAE: 0.1950
Test RMSE: 0.3042
Test R²: 0.9696
Test MAE: 0.1806


### 2) V modeling: linear baseline vs Random Forest

Train the linear baseline for inter-site interaction `V`, then fit a tuned Random Forest on RF-safe matrices (rebuilt from raw processed tables to avoid in-memory mutation effects).

Use this block to:
- compare linear vs non-linear predictive behavior for `V`,
- export aligned prediction files for both models,
- save feature-importance outputs for downstream interpretation.


In [154]:

# -----------------------------
# Step 0: Reset indices to ensure alignment
# -----------------------------
X_train_v = X_train_v.reset_index(drop=True)
X_test_v = X_test_v.reset_index(drop=True)
y_train_v = y_train_v.reset_index(drop=True)
y_test_v = y_test_v.reset_index(drop=True)

# Add intercept for statsmodels
X_train_v = sm.add_constant(X_train_v)
X_test_v = sm.add_constant(X_test_v)

# -----------------------------
# Step 2: Fit initial OLS model
# -----------------------------
modelo_v = sm.OLS(y_train_v, X_train_v).fit()
print("Initial OLS Model:")
print(modelo_v.summary())

# -----------------------------
# Step 3: Stepwise selection (p-value threshold 0.05)
# -----------------------------
modelo_step_v = stepwise(modelo_v, pvalue_limit=0.05)
print("Stepwise-selected Model:")
print(modelo_step_v.summary())

# -----------------------------
# Step 4: Residual normality test (Shapiro-Francia)
# -----------------------------
teste_sf = shapiro_francia(modelo_step_v.resid)
teste_sf_items = list(teste_sf.items())
method, statistics_W, statistics_z, p = teste_sf_items
print('Shapiro-Francia Test: Statistics W=%.5f, p-value=%.6f' % (statistics_W[1], p[1]))

alpha = 0.05
if p[1] > alpha:
    print('Reject H0 - Normal distribution')
else:
    print('Fail to reject H0 - Non-normal distribution')



Initial OLS Model:
                            OLS Regression Results                            
Dep. Variable:                      V   R-squared:                       0.234
Model:                            OLS   Adj. R-squared:                  0.233
Method:                 Least Squares   F-statistic:                     940.2
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:49:24   Log-Likelihood:             1.0257e+05
No. Observations:              178924   AIC:                        -2.050e+05
Df Residuals:                  178865   BIC:                        -2.044e+05
Df Model:                          58                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0676   

In [155]:
def _exog_name_to_df_col(name: str) -> str:
    clean = str(name).replace("Q('", "").replace("')", "")
    if clean == "Intercept":
        return "const"
    return clean

selected_cols_v = [_exog_name_to_df_col(n) for n in modelo_step_v.model.exog_names]

X_train_v_sel = X_train_v[selected_cols_v].copy()
X_test_v_sel = X_test_v[selected_cols_v].copy()

metrics_v = evaluate_regression(
    modelo_step_v,
    X_train_v_sel,
    y_train_v,
    X_test_v_sel,
    y_test_v,
)

Training RMSE: 0.1364
Training R²: 0.2336
Training MAE: 0.0464
Test RMSE: 0.0988
Test R²: 0.1673
Test MAE: 0.0468


In [156]:
# Rebuild V matrices for tree models (avoid mutations from linear-model cells)
v_train_for_dummies_rf = df_v_train.drop(columns=['qmof_id'])
v_test_for_dummies_rf = df_v_test.drop(columns=['qmof_id'])

v_train_dummies_rf = pd.get_dummies(v_train_for_dummies_rf, columns=['atom_i', 'atom_j'], dtype=int, drop_first=True)
v_test_dummies_rf = pd.get_dummies(v_test_for_dummies_rf, columns=['atom_i', 'atom_j'], dtype=int, drop_first=True)

target_v_rf = 'V'
predictor_cols_v_rf = [c for c in v_train_dummies_rf.columns if c != target_v_rf]

X_train_v_rf = v_train_dummies_rf[predictor_cols_v_rf].copy()
y_train_v_rf = v_train_dummies_rf[target_v_rf].copy().reset_index(drop=True)
X_test_v_rf = v_test_dummies_rf.reindex(columns=predictor_cols_v_rf, fill_value=0).copy()
y_test_v_rf = v_test_dummies_rf[target_v_rf].copy().reset_index(drop=True)

print("\nFor predicting V (RF-safe matrices):")
print(f"Shape of X train data: {X_train_v_rf.shape}")
print(f"Shape of y train data: {y_train_v_rf.shape}")
print(f"Shape of X test data: {X_test_v_rf.shape}")
print(f"Shape of y test data: {y_test_v_rf.shape}")



For predicting V (RF-safe matrices):
Shape of X train data: (178924, 58)
Shape of y train data: (178924,)
Shape of X test data: (40927, 58)
Shape of y test data: (40927,)


In [157]:

param_grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10],
    "max_features": ["sqrt", 0.6],
    "min_samples_leaf": [10, 30],
}

rf_grid_v = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring="neg_mean_squared_error",
    cv=3,
    verbose=1,
    n_jobs=-1,
)


rf_grid_v.fit(X_train_v_rf, y_train_v_rf)




Fitting 3 folds for each of 16 candidates, totalling 48 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...dom_state=100)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [5, 10], 'max_features': ['sqrt', 0.6], 'min_samples_leaf': [10, 30], 'n_estimators': [100, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and p

In [158]:
print("Best RF params (V):", rf_grid_v.best_params_)

rf_v_best = rf_grid_v.best_estimator_

def evaluate_sklearn_regressor(model, X_train, y_train, X_test, y_test, name="model"):
    """Print and return RMSE, MAE and R² for train and test sets."""
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    mse_train = mean_squared_error(y_train, y_pred_train)
    mse_test = mean_squared_error(y_test, y_pred_test)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    mae_test = mean_absolute_error(y_test, y_pred_test)

    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    print(f"=== {name} ===")
    print("Train -> RMSE: {:.4f}, MAE: {:.4f}, R²: {:.4f}".format(np.sqrt(mse_train), mae_train, r2_train))
    print("Test  -> RMSE: {:.4f}, MAE: {:.4f}, R²: {:.4f}".format(np.sqrt(mse_test), mae_test, r2_test))

    return {
        "rmse_train": np.sqrt(mse_train),
        "rmse_test": np.sqrt(mse_test),
        "mae_train": mae_train,
        "mae_test": mae_test,
        "r2_train": r2_train,
        "r2_test": r2_test,
    }

metrics_rf_v_best = evaluate_regression(rf_v_best, X_train_v_rf, y_train_v_rf, X_test_v_rf, y_test_v_rf)

# Variable importance for U and V
rf_features_v = pd.DataFrame({
    "feature": X_train_v_rf.columns,
    "importance": rf_v_best.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

# Save RF feature importance artifact for 04_results_evaluation.ipynb
out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)
rf_features_v.to_csv(out_dir / 'rf_v_feature_importance.csv', index=False)

print(f"Saved: {out_dir / 'rf_v_feature_importance.csv'} ({len(rf_features_v)} rows)")
display(rf_features_v.head(20))

Best RF params (V): {'max_depth': 10, 'max_features': 0.6, 'min_samples_leaf': 30, 'n_estimators': 100}
Training RMSE: 0.0535
Training R²: 0.8821
Training MAE: 0.0248
Test RMSE: 0.0505
Test R²: 0.7822
Test MAE: 0.0263
Saved: ../data/processed/rf_v_feature_importance.csv (58 rows)


,feature,importance
0,volume,0.283076
1,atom_j_Cu,0.172234
2,distance,0.111097
3,ddec_charge_j,0.090940
4,atom_i_Cu,0.082911
5,atom_j_S,0.075124
6,pld,0.067022
7,ddec_charge_i,0.033231
8,atom_i_S,0.027993
9,atom_j_N,0.021866


In [159]:
metrics_v = evaluate_regression(
    rf_v_best,
    X_train_v_rf,
    y_train_v_rf,
    X_test_v_rf,
    y_test_v_rf,
)

Training RMSE: 0.0535
Training R²: 0.8821
Training MAE: 0.0248
Test RMSE: 0.0505
Test R²: 0.7822
Test MAE: 0.0263


In [160]:
from pathlib import Path

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

def _predict_identity(model, X):
    # Identity inverse transform (current notebook uses lmbda=None for U and V)
    return pd.Series(model.predict(X)).reset_index(drop=True)

# U predictions
X_u = X_test_u_sel if "X_test_u_sel" in globals() else X_test_u
y_u_true = y_test_u.reset_index(drop=True)
y_u_pred = _predict_identity(modelo_step_u, X_u)

pred_u = pd.DataFrame({
    "qmof_id": df_u_test["qmof_id"].reset_index(drop=True) if "qmof_id" in df_u_test.columns else pd.Series(range(len(y_u_true))),
    "target": "U",
    "actual": y_u_true,
    "predicted": y_u_pred,
    "residual": y_u_true - y_u_pred,
    "atom_i": df_u_test["atom_i"].reset_index(drop=True) if "atom_i" in df_u_test.columns else pd.NA,
})
pred_u.to_csv(out_dir / "predictions_u_linear.csv", index=False)

# V predictions
X_v = X_test_v_sel if "X_test_v_sel" in globals() else X_test_v_rf
y_v_true = y_test_v.reset_index(drop=True) if "X_test_v_sel" in globals() else y_test_v_rf.reset_index(drop=True)
y_v_pred = _predict_identity(modelo_step_v, X_v)

pred_v = pd.DataFrame({
    "qmof_id": df_v_test["qmof_id"].reset_index(drop=True) if "qmof_id" in df_v_test.columns else pd.Series(range(len(y_v_true))),
    "target": "V",
    "actual": y_v_true,
    "predicted": y_v_pred,
    "residual": y_v_true - y_v_pred,
    "atom_i": df_v_test["atom_i"].reset_index(drop=True) if "atom_i" in df_v_test.columns else pd.NA,
    "atom_j": df_v_test["atom_j"].reset_index(drop=True) if "atom_j" in df_v_test.columns else pd.NA,
})
pred_v.to_csv(out_dir / "predictions_v_linear.csv", index=False)

# Stepwise OLS coefficients (for evaluation notebook right panel)
pd.DataFrame(
    {"feature": modelo_step_u.params.index.astype(str), "coefficient": modelo_step_u.params.values}
).to_csv(out_dir / "linear_u_coefficients.csv", index=False)
pd.DataFrame(
    {"feature": modelo_step_v.params.index.astype(str), "coefficient": modelo_step_v.params.values}
).to_csv(out_dir / "linear_v_coefficients.csv", index=False)

# V predictions (Random Forest)
X_v_rf = X_test_v_rf
y_v_rf_true = y_test_v_rf.reset_index(drop=True)
y_v_rf_pred = _predict_identity(rf_v_best, X_v_rf)

pred_v_rf = pd.DataFrame({
    "qmof_id": df_v_test["qmof_id"].reset_index(drop=True) if "qmof_id" in df_v_test.columns else pd.Series(range(len(y_v_rf_true))),
    "model": "random_forest",
    "target": "V",
    "actual": y_v_rf_true,
    "predicted": y_v_rf_pred,
    "residual": y_v_rf_true - y_v_rf_pred,
    "atom_i": df_v_test["atom_i"].reset_index(drop=True) if "atom_i" in df_v_test.columns else pd.NA,
    "atom_j": df_v_test["atom_j"].reset_index(drop=True) if "atom_j" in df_v_test.columns else pd.NA,
})
pred_v_rf.to_csv(out_dir / "predictions_v_rf.csv", index=False)

print(f"Saved: {out_dir / 'predictions_u_linear.csv'} ({len(pred_u)} rows)")
print(f"Saved: {out_dir / 'predictions_v_linear.csv'} ({len(pred_v)} rows)")
print(f"Saved: {out_dir / 'linear_u_coefficients.csv'} ({len(modelo_step_u.params)} terms)")
print(f"Saved: {out_dir / 'linear_v_coefficients.csv'} ({len(modelo_step_v.params)} terms)")
print(f"Saved: {out_dir / 'predictions_v_rf.csv'} ({len(pred_v_rf)} rows)")

Saved: ../data/processed/predictions_u_linear.csv (1658 rows)
Saved: ../data/processed/predictions_v_linear.csv (40927 rows)
Saved: ../data/processed/linear_u_coefficients.csv (31 terms)
Saved: ../data/processed/linear_v_coefficients.csv (33 terms)
Saved: ../data/processed/predictions_v_rf.csv (40927 rows)
